In [ ]:

# PROJECT: EARLY DETECTION OF LIQUID LOADING IN GAS WELLS

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. CREATE SYNTHETIC GAS-WELL PRODUCTION DATA
np.random.seed(42)
n = 180

df = pd.DataFrame({
    "Day": np.arange(1, n + 1),
    "Gas_Rate_std_m3_day": np.linspace(100000, 20000, n) * np.random.normal(1.0, 0.025, n),
    "Pressure_bar_abs": np.linspace(80, 45, n),
    "Temperature_K": np.linspace(320, 315, n),
    "Tubing_ID_mm": np.full(n, 62.0),
    "Z": np.linspace(0.90, 0.92, n),
    "Liquid_Density_kg_m3": np.full(n, 1000.0),      # Water
    "Surface_Tension_N_m": np.full(n, 0.030)         # Water-gas
})

df["Gas_Rate_std_m3_day"] = df["Gas_Rate_std_m3_day"].clip(lower=100)

# 2. PHYSICAL CONSTANTS & UNIT CONVERSIONS

R = 8.314462618        # J/(mol K)
MW_GAS = 0.020         # kg/mol (~0.69 gas gravity)
P_STD = 101325.0       # Pa
T_STD = 288.15         # K (15 °C)
Z_STD = 1.0

# SI constant for Turner droplet model: ~4.45
TURNER_C = 4.45

df["Pressure_Pa"] = df["Pressure_bar_abs"] * 1e5
df["Tubing_ID_m"] = df["Tubing_ID_mm"] / 1000.0
df["Gas_Rate_std_m3_s"] = df["Gas_Rate_std_m3_day"] / 86400.0

# 3. VELOCITY & TURNER CALCULATIONS (SI)

# Gas density at in-situ conditions
df["Gas_Density_kg_m3"] = (df["Pressure_Pa"] * MW_GAS) / (df["Z"] * R * df["Temperature_K"])

# In-situ volumetric rate & velocity
df["Gas_Rate_Flowing_m3_s"] = (
    df["Gas_Rate_std_m3_s"] * (P_STD / df["Pressure_Pa"]) * (df["Temperature_K"] / T_STD) * (df["Z"] / Z_STD)
)
df["Tubing_Area_m2"] = np.pi * (df["Tubing_ID_m"] ** 2) / 4.0
df["Actual_Gas_Velocity_m_s"] = df["Gas_Rate_Flowing_m3_s"] / df["Tubing_Area_m2"]

# Turner critical velocity
density_diff = df["Liquid_Density_kg_m3"] - df["Gas_Density_kg_m3"]
turner_term = (df["Surface_Tension_N_m"] * density_diff) / (df["Gas_Density_kg_m3"] ** 2)
df["Critical_Gas_Velocity_m_s"] = TURNER_C * (turner_term ** 0.25)

# Critical standard gas rate
df["Critical_Gas_Rate_Flowing_m3_s"] = df["Critical_Gas_Velocity_m_s"] * df["Tubing_Area_m2"]
df["Critical_Gas_Rate_std_m3_day"] = (
    df["Critical_Gas_Rate_Flowing_m3_s"] * (df["Pressure_Pa"] / P_STD) * (T_STD / df["Temperature_K"]) * (Z_STD / df["Z"]) * 86400.0
)


# 4. DIAGNOSTIC RATIOS & RISK SCREENING

df["Velocity_Ratio"] = df["Actual_Gas_Velocity_m_s"] / df["Critical_Gas_Velocity_m_s"]
df["Below_Critical_Velocity"] = df["Velocity_Ratio"] < 1.0

conditions = [
    df["Velocity_Ratio"] < 0.8,
    df["Velocity_Ratio"] < 1.0,
    df["Velocity_Ratio"] < 1.2
]
choices = ["High screening risk", "Elevated screening risk", "Watch"]
df["Risk_Category"] = np.select(conditions, choices, default="Lower screening risk")


# 5. MULTI-PANEL DASHBOARD PLOT

fig, axs = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

# Panel 1: Production Rate
axs[0, 0].plot(df["Day"], df["Gas_Rate_std_m3_day"] / 1000.0, color="navy", lw=2, label="Actual Rate")
axs[0, 0].set_title("Standard Gas Production History", fontweight="bold")
axs[0, 0].set_xlabel("Day")
axs[0, 0].set_ylabel("Gas Rate (k m³/day)")
axs[0, 0].grid(True, linestyle="--", alpha=0.6)
axs[0, 0].legend()

# Panel 2: In-situ Velocities
axs[0, 1].plot(df["Day"], df["Actual_Gas_Velocity_m_s"], color="blue", label="Actual Gas Velocity")
axs[0, 1].plot(df["Day"], df["Critical_Gas_Velocity_m_s"], color="crimson", linestyle="--", label="Turner Critical Velocity")
axs[0, 1].set_title("Actual vs. Critical Gas Velocity (SI)", fontweight="bold")
axs[0, 1].set_xlabel("Day")
axs[0, 1].set_ylabel("Velocity (m/s)")
axs[0, 1].grid(True, linestyle="--", alpha=0.6)
axs[0, 1].legend()

# Panel 3: Velocity Ratio (Screening Indicator)
axs[1, 0].plot(df["Day"], df["Velocity_Ratio"], color="purple", lw=2, label="Velocity Ratio (R)")
axs[1, 0].axhline(1.0, color="red", linestyle="--", lw=1.5, label="Loading Boundary (R = 1.0)")
axs[1, 0].fill_between(df["Day"], 0, 1.0, color="red", alpha=0.12, label="Liquid Loaded Region")
axs[1, 0].set_title("Velocity Ratio / Loading Indicator", fontweight="bold")
axs[1, 0].set_xlabel("Day")
axs[1, 0].set_ylabel("Ratio (dimensionless)")
axs[1, 0].set_ylim(bottom=0)
axs[1, 0].grid(True, linestyle="--", alpha=0.6)
axs[1, 0].legend(loc="lower left")

# Panel 4: Critical Gas Rate Comparison
axs[1, 1].plot(df["Day"], df["Gas_Rate_std_m3_day"] / 1000.0, color="navy", lw=2, label="Actual Rate")
axs[1, 1].plot(df["Day"], df["Critical_Gas_Rate_std_m3_day"] / 1000.0, color="red", linestyle="--", lw=2, label="Critical Rate ($q_{crit}$)")
axs[1, 1].set_title("Actual vs. Critical Production Rate", fontweight="bold")
axs[1, 1].set_xlabel("Day")
axs[1, 1].set_ylabel("Gas Rate (k m³/day)")
axs[1, 1].grid(True, linestyle="--", alpha=0.6)
axs[1, 1].legend()

plt.suptitle("Gas Well Liquid Loading Diagnostic Dashboard (Turner Method)", fontsize=14, fontweight="bold")
plt.show()

# First loading day output
below = df[df["Below_Critical_Velocity"]]
if not below.empty:
    print(f"Liquid Loading Onset: Day {int(below.iloc[0]['Day'])} (Velocity Ratio = {below.iloc[0]['Velocity_Ratio']:.3f})")